# 05 — Query ohsome API for all Petrópolis 1 km cells

Production notebook for the 895 validated 1 km cells intersecting Petrópolis. It reuses the response structure validated in Notebook 04 and queries four endpoints in conservative batches with cache, retry/backoff, checkpoints, raw-response persistence, and final integrity checks.


In [ ]:
from pathlib import Path
import json, time, math
from datetime import datetime, timezone
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import mapping

BASE_URL = 'https://api.ohsome.org/v1'
FILTER = 'highway=* and geometry:line'
BATCH_SIZE = 50
MAX_RETRIES = 5
TIMEOUT = 600


In [ ]:
def find_root(start=Path.cwd()):
    for p in [start.resolve(), *start.resolve().parents]:
        if (p/'notebooks').exists() and (p/'Petropolis').exists(): return p
    raise FileNotFoundError('Repository root not found')
ROOT=find_root()
GRID_FILE=ROOT/'data/processed/Petropolis/petropolis_grid_1km.parquet'
RAW_DIR=ROOT/'data/raw/ohsome/Petropolis/all_895_cells'
OUT_DIR=ROOT/'data/processed/Petropolis'
RAW_DIR.mkdir(parents=True, exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)
grid=gpd.read_parquet(GRID_FILE)
assert len(grid)==895, f'Expected 895 cells, found {len(grid)}'
assert grid['grid_1km_id'].is_unique and grid['grid_1km_id'].notna().all()
assert grid.crs.to_epsg()==4674
query_grid=grid.to_crs(4326).copy()
print('Cells:', len(query_grid))


## 1. Metadata and temporal extent


In [ ]:
session=requests.Session()
meta_path=RAW_DIR/'ohsome_metadata.json'
if meta_path.exists():
    metadata=json.loads(meta_path.read_text(encoding='utf-8'))
else:
    r=session.get(f'{BASE_URL}/metadata', timeout=60); r.raise_for_status(); metadata=r.json(); meta_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
api_version=metadata.get('apiVersion')
temporal=metadata['extractRegion']['temporalExtent']
t0,t1=temporal['fromTimestamp'],temporal['toTimestamp']
print('API version:', api_version); print('Temporal extent:', t0, '→', t1)


In [ ]:
def fc_for(gdf):
    return {'type':'FeatureCollection','features':[{'type':'Feature','properties':{'id':str(r.grid_1km_id)},'geometry':mapping(r.geometry)} for r in gdf[['grid_1km_id','geometry']].itertuples(index=False)]}

def request_with_retry(endpoint, data):
    last=None
    for attempt in range(MAX_RETRIES):
        try:
            resp=session.post(f'{BASE_URL}{endpoint}', data=data, timeout=TIMEOUT)
            if resp.status_code in (429,500,502,503,504):
                raise RuntimeError(f'HTTP {resp.status_code}: {resp.text[:300]}')
            resp.raise_for_status(); body=resp.json()
            if 'groupByResult' not in body: raise ValueError(f'Unexpected response keys: {list(body)}')
            return resp.status_code, body
        except Exception as e:
            last=e
            if attempt==MAX_RETRIES-1: raise
            time.sleep(min(60, 2**attempt*3))
    raise last

def parse_groupby(body, expected_ids):
    rows=[]
    for group in body['groupByResult']:
        gid=str(group['groupByObject']); result=group.get('result',[])
        value=None if not result else result[0].get('value')
        rows.append((gid,value))
    df=pd.DataFrame(rows,columns=['grid_1km_id','value'])
    assert df['grid_1km_id'].is_unique
    missing=set(expected_ids)-set(df['grid_1km_id'])
    extra=set(df['grid_1km_id'])-set(expected_ids)
    if missing or extra: raise ValueError(f'Boundary mismatch: missing={len(missing)} extra={len(extra)}')
    return df


## 2. Batch production queries


In [ ]:
ENDPOINTS={
 'road_length_m':('/elements/length/groupBy/boundary',{}),
 'road_element_count':('/elements/count/groupBy/boundary',{}),
 'contributors_count':('/users/count/groupBy/boundary',{'time':f'{t0}/{t1}'}),
 'contributions_count':('/contributions/count/groupBy/boundary',{'time':f'{t0},{t1}'})
}
ids=query_grid['grid_1km_id'].astype(str).tolist()
n_batches=math.ceil(len(query_grid)/BATCH_SIZE)
results={}
manifest=[]
for metric,(endpoint,extra) in ENDPOINTS.items():
    metric_frames=[]
    metric_dir=RAW_DIR/metric; metric_dir.mkdir(exist_ok=True)
    for b in range(n_batches):
        batch=query_grid.iloc[b*BATCH_SIZE:(b+1)*BATCH_SIZE].copy()
        expected=batch['grid_1km_id'].astype(str).tolist()
        cache=metric_dir/f'batch_{b+1:03d}.json'
        if cache.exists():
            record=json.loads(cache.read_text(encoding='utf-8')); body=record['body']; source='cache'
        else:
            data={'filter':FILTER,'format':'json','bpolys':json.dumps(fc_for(batch),separators=(',',':'))}; data.update(extra)
            status,body=request_with_retry(endpoint,data)
            record={'metric':metric,'endpoint':endpoint,'batch':b+1,'status_code':status,'apiVersion':body.get('apiVersion'),'recorded_at_utc':datetime.now(timezone.utc).isoformat(),'body':body}
            cache.write_text(json.dumps(record,indent=2),encoding='utf-8'); source='api'
        parsed=parse_groupby(body,expected).rename(columns={'value':metric})
        metric_frames.append(parsed)
        manifest.append({'metric':metric,'batch':b+1,'n_expected':len(expected),'n_returned':len(parsed),'source':source})
        print(metric, f'batch {b+1}/{n_batches}', source, len(parsed))
    metric_df=pd.concat(metric_frames,ignore_index=True)
    assert len(metric_df)==895 and metric_df['grid_1km_id'].is_unique
    results[metric]=metric_df
manifest_df=pd.DataFrame(manifest)
display(manifest_df.groupby('metric')[['n_expected','n_returned']].sum())


## 3. Consolidate and validate


In [ ]:
out=grid.copy()
for metric,dfm in results.items(): out=out.merge(dfm,on='grid_1km_id',how='left',validate='one_to_one')
out['road_length_km']=out['road_length_m']/1000.0
out['road_density_km_per_km2']=out['road_length_km']/(out['cell_area_m2']/1_000_000.0)
out['ohsome_api_version']=api_version
out['oshdb_from_timestamp']=t0
out['oshdb_to_timestamp']=t1
out['ohsome_filter']=FILTER
checks={
 'rows_895':len(out)==895,
 'unique_grid_id':out['grid_1km_id'].is_unique,
 'geometry_valid':out.geometry.is_valid.all(),
 'no_missing_length':out['road_length_m'].notna().all(),
 'no_missing_element_count':out['road_element_count'].notna().all(),
 'no_missing_contributors':out['contributors_count'].notna().all(),
 'no_missing_contributions':out['contributions_count'].notna().all(),
 'nonnegative_metrics':(out[['road_length_m','road_element_count','contributors_count','contributions_count']]>=0).all().all()
}
display(pd.Series(checks,name='PASS'))
if not all(checks.values()): raise ValueError([k for k,v in checks.items() if not v])
OUTPUT=OUT_DIR/'petropolis_ohsome_1km_895_cells.parquet'
out.to_parquet(OUTPUT,index=False)
manifest_df.to_csv(RAW_DIR/'batch_manifest.csv',index=False)
summary=pd.DataFrame({m:out[m].describe() for m in ['road_length_m','road_element_count','contributors_count','contributions_count']})
display(summary)
print('Saved:',OUTPUT)
